In [ ]:
# %pip install langchian \
    # langchain_community \
    # transformers \
    # datasets \
    # sentence-transformers \
    # langchain-huggingface \
    # faiss-cpu

In [9]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, pipeline, AutoModelForQuestionAnswering


In [2]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

In [3]:
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

data[:2]

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'),
 Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

In [11]:

# 1️⃣ Embedding model (for FAISS)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# If you already have a saved FAISS index:

vectorstore = FAISS.from_documents(docs, embedding_model)

In [12]:

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # top 3 docs

In [13]:
# 2️⃣ Load TinyBERT QA model
tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
model = AutoModelForQuestionAnswering.from_pretrained("Intel/dynamic_tinybert")

qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)


Device set to use cpu


Invalid model-index. Not loading eval results into CardData.


In [14]:
# 3️⃣ Helper function to run RAG QA
def rag_answer(question: str, retriever, qa_pipeline, k=3):
    # Retrieve top-k relevant documents
    docs = retriever.get_relevant_documents(question)
    
    # Combine their content into one context string
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Feed question + context into TinyBERT QA pipeline
    result = qa_pipeline(question=question, context=context)
    
    return result["answer"], docs

In [15]:
# 4️⃣ Ask a question
query = "What is Databricks used for?"
answer, docs = rag_answer(query, retriever, qa_pipeline, k=3)

print("Answer:", answer)
print("\nRetrieved docs:")
for i, doc in enumerate(docs):
    print(f"Doc {i+1}:\n{doc.page_content}\n")

C:\Users\rouna\AppData\Local\Temp\ipykernel_23248\2799447117.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(question)


Answer: large-scale data processing

Retrieved docs:
Doc 1:
"Apache Spark has its architectural foundation in the resilient distributed dataset (RDD), a read-only multiset of data items distributed over a cluster of machines, that is maintained in a fault-tolerant way. The Dataframe API was released as an abstraction on top of the RDD, followed by the Dataset API. In Spark 1.x, the RDD was the primary application programming interface (API), but as of Spark 2.x use of the Dataset API is encouraged even though the RDD API is not deprecated. The RDD technology still underlies the Dataset API."

Doc 2:
"Apache Spark is an open-source unified analytics engine for large-scale data processing. Spark provides an interface for programming clusters with implicit data parallelism and fault tolerance. Originally developed at the University of California, Berkeley's AMPLab, the Spark codebase was later donated to the Apache Software Foundation, which has maintained it since."

Doc 3:
"According to